<a href="https://colab.research.google.com/github/chaithu-gif/AI_LAB/blob/main/week_8ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
def parse(s):
    name, args = s.strip().split("(")
    return name.strip(), tuple(x.strip() for x in args.rstrip(")").split(","))


def unify(x, y, theta):
    if x.startswith("?"):
        if x in theta:
            return unify(theta[x], y, theta)
        if y.startswith("?") and y in theta:
            return unify(x, theta[y], theta)
        if x != y:
            theta[x] = y
        return theta
    if y.startswith("?"):
        return unify(y, x, theta)
    return theta if x == y else None


def match(p, f, theta):
    if p[0] != f[0] or len(p[1]) != len(f[1]):
        return None

    for x, y in zip(p[1], f[1]):
        theta = unify(x, y, theta)
        if theta is None:
            return None

    return theta


def subst(theta, term):
    while term in theta:
        term = theta[term]
    return term


def forward_chaining(facts, rules, query):
    inferred = {}
    query_parsed = parse(query)

    def find_goal(goal, visited=None):
        if visited is None:
            visited = set()

        if goal in visited:
            return None
        visited.add(goal)

        if goal in facts_initial:
            return [goal], []

        if goal in inferred:
            premises, theta, rule = inferred[goal]
            used_facts = []
            used_rules = []

            for p in premises:
                inst_p = (p[0], tuple(subst(theta, x) for x in p[1]))
                result = find_goal(inst_p, visited.copy())
                if result:
                    used_facts.extend(result[0])
                    used_rules.extend(result[1])

            used_rules.append(rule)
            return list(dict.fromkeys(used_facts)), list(dict.fromkeys(used_rules))

        return None

    facts_initial = set(facts)
    all_facts = set(facts)

    while True:
        new = []

        for premises, conclusion, original in rules:
            substitutions = [{}]

            for p in premises:
                updated = []
                for theta in substitutions:
                    for fact in all_facts:
                        # Apply current substitutions to premise before matching
                        inst_p = (p[0], tuple(subst(theta, x) for x in p[1]))
                        temp = match(inst_p, fact, theta.copy())
                        if temp is not None:
                            updated.append(temp)
                substitutions = updated

            for theta in substitutions:
                result = (
                    conclusion[0],
                    tuple(subst(theta, x) for x in conclusion[1])
                )

                if result not in all_facts and result not in new:
                    new.append(result)
                    inferred[result] = (premises, theta, original)

        for fact in new:
            all_facts.add(fact)

            if match(query_parsed, fact, {}) is not None:
                result = find_goal(fact)
                return True, result, new

        if not new:
            break

    for fact in all_facts:
        if match(query_parsed, fact, {}) is not None:
            return True, find_goal(fact), list(inferred.keys())

    return False, None, list(inferred.keys())


# User Input
facts = set()
rules = []

n = int(input("Enter number of facts: "))

for i in range(n):
    facts.add(parse(input("Enter fact: ")))

m = int(input("Enter number of rules: "))

for i in range(m):
    rule = input("Enter rule (A(?x) AND B(?x) -> C(?x)): ")
    left, right = rule.split("->")
    premises = [parse(x) for x in left.split("AND")]
    conclusion = parse(right)
    rules.append((premises, conclusion, rule))

query = input("Enter query: ")

print("\n--- FACTS USED ---")
for f in facts:
    print(f)

print("\n--- RULES ENTERED ---")
for r in rules:
    print(r[2])

result, proof, inferred = forward_chaining(facts, rules, query)

print("\n--- INFERRED FACTS ---")
for f in inferred:
    print(f)

print("\n--- RESULT ---")

if result:
    print("Query:", query)
    print("TRUE: Query is proved")

    print("\nFacts used in proof:")
    for f in proof[0]:
        print(f"{f[0]}({', '.join(f[1])})")

    print("\nRules used in proof:")
    for r in proof[1]:
        print(r)
else:
    print("FALSE: Query cannot be proved")

Enter number of facts: 2
Enter fact: Human(Ram)
Enter fact: Human(John)
Enter number of rules: 2
Enter rule (A(?x) AND B(?x) -> C(?x)): Human(?x)->Mortal(?x)
Enter rule (A(?x) AND B(?x) -> C(?x)): Mortal(?x)->Living(?x)
Enter query: Living(Ram)

--- FACTS USED ---
('Human', ('John',))
('Human', ('Ram',))

--- RULES ENTERED ---
Human(?x)->Mortal(?x)
Mortal(?x)->Living(?x)

--- INFERRED FACTS ---
('Living', ('Ram',))
('Living', ('John',))

--- RESULT ---
Query: Living(Ram)
TRUE: Query is proved

Facts used in proof:
Human(Ram)

Rules used in proof:
Human(?x)->Mortal(?x)
Mortal(?x)->Living(?x)
